# 02 - Evaluation Scoring (Multi-Endpoint Load Balanced)

**Configuration**:
- **2x Glider Endpoints**: `localhost:8807`, `localhost:8808`
- **2x LLama Endpoints**: `localhost:8806`, `localhost:8809`
- **Pipeline**: Distributes requests round-robin across these endpoints.
- **Parallelism**: High concurrency to maximize throughput.

In [1]:
# === Path Setup ===
import sys
from pathlib import Path

# Notebook paths - all ares notebooks are in artemis_final/notebooks/ares/
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent  # artemis_final/
ROOT_DIR = ARTEMIS_DIR.parent             # Which_VLM_Router/

# Add to sys.path for imports
for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f" ARTEMIS_DIR: {ARTEMIS_DIR}")

# Cell 1: Setup
%load_ext autoreload
%autoreload 2


import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)-15s | %(message)s', datefmt='%H:%M:%S')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)

📁 ARTEMIS_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final


In [2]:
# Cell 2: Database Connection
from sqlalchemy import text
from ares.db.connection import get_engine

engine = get_engine()
print("Connected to DB")

# Apply migrations
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_score FLOAT"))
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_rank_group INTEGER"))
    conn.execute(text("ALTER TABLE vlm_evaluations ADD COLUMN IF NOT EXISTS judge_molmo_raw TEXT"))
    conn.commit()
print("Migrations applied")

Connected to DB
Migrations applied


In [3]:
# Cell 3: Configure Endpoints
from inference_engine.runners import OpenAIStyleRunner
from inference_engine.config import ModelEndpoint

# Your current vLLM setup:
# - Glider: ports 8810, 8811
# - Llama Scout: ports 8812, 8813

endpoints = [

    # Glider (text-only LLM judge) - Load Balanced
    ModelEndpoint(
        name="glider",
        model_id="PatronusAI/glider",
        base_url="http://localhost:8810/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    ModelEndpoint(
        name="glider",
        model_id="PatronusAI/glider",
        base_url="http://localhost:8811/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    # Llama Scout (VLM judge with image) - Load Balanced
    ModelEndpoint(
        name="vlm_judge",
        model_id="nvidia/Llama-4-Scout-17B-16E-Instruct-FP8",
        base_url="http://localhost:8812/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),
    ModelEndpoint(
        name="vlm_judge",
        model_id="nvidia/Llama-4-Scout-17B-16E-Instruct-FP8",
        base_url="http://localhost:8813/v1",
        api_key="EMPTY",
        pricing={},
        extra_params={}
    ),

]

runner = OpenAIStyleRunner(
    models=endpoints,
    request_timeout_s=180,
    max_workers=128
)

print(f"Configured {len(endpoints)} endpoints:")
for ep in endpoints:
    print(f"  - {ep.name}: {ep.base_url}")


Configured 4 endpoints:
  - glider: http://localhost:8810/v1
  - glider: http://localhost:8811/v1
  - vlm_judge: http://localhost:8812/v1
  - vlm_judge: http://localhost:8813/v1


In [4]:
# Cell 4: Initialize Pipeline
from ares.evaluation.router_eval_pipeline import RouterEvalPipeline

pipeline = RouterEvalPipeline(
    engine=engine,
    runner=runner,
    glider_model_names=["glider"],
    vlm_judge_model_names=["vlm_judge"],
    tracker_path="eval_progress.json",
    use_glider=True,
    use_vlm_judge=True,
)

print("Pipeline initialized!")
print(f"Glider models: {pipeline.glider_model_names}")
print(f"VLM Judge models: {pipeline.vlm_judge_model_names}")

Pipeline initialized!
Glider models: ['glider']
VLM Judge models: ['vlm_judge']


In [6]:
# Cell 5: Run Evaluation
# This will:
# 1. Load samples per source_config
# 2. Compute static metrics (exact match, F1, etc.)
# 3. Compute confidence scores
# 4. Run Glider (text evaluator) on 2 GPUs
# 5. Run Llama Scout (VLM judge with image) on 2 GPUs
# 6. Write all results to vlm_evaluations table

# Reset ALL progress
# pipeline.reset_progress()

pipeline.evaluate_all(
    batch_size=100,
    force=False,             # Set True to recompute all
    max_parallel_configs=2, # Process 2 source_configs at a time
    split=None,              # Filter: 'train', 'val', 'test', or None
)

06:57:41 | EVAL_PIPELINE   | ============================================================
06:57:41 | EVAL_PIPELINE   | EVALUATION PIPELINE
06:57:41 | EVAL_PIPELINE   | ============================================================
06:57:41 | EVAL_PIPELINE   | Source configs: 48
06:57:41 | EVAL_PIPELINE   | Previous progress: 40500 samples
06:57:41 | EVAL_PIPELINE   | Use Glider: True, Use VLM Judge: True
06:57:41 | EVAL_PIPELINE   | Force recompute: False
06:57:41 | EVAL_PIPELINE   | Split filter: ALL
06:57:41 | EVAL_PIPELINE   | ============================================================


[aokvqa]:   0%|          | 0/2000 [00:00<?, ?s/s]

[ai2d]:   0%|          | 0/2000 [00:00<?, ?s/s]

[chart2text]:   0%|          | 0/1999 [00:00<?, ?s/s]

[chartqa]:   0%|          | 0/2000 [00:00<?, ?s/s]

...

[vqav2]:   0%|          | 0/1000 [00:00<?, ?s/s]

[vsr]:   0%|          | 0/1000 [00:00<?, ?s/s]

10:50:59 | EVAL_PIPELINE   | VLM Judge error for vqav2_206_d3cc0e1d: unhashable type: 'list'
10:54:28 | VLM_JUDGE       | JSON parse failed, trying regex. Raw: {
  "scores": {
    "A": 10,
    "B": 10,
    "C": 0,
    "D": 0
  },
  "ranking": [
    ["A", "B"],
    ["E is not an option but if it was it would be in this group",],
    ["C", "D"]
  ]
}...
10:54:28 | VLM_JUDGE       | Extracted scores via regex: {'A': 10.0, 'B': 10.0, 'C': 0.0, 'D': 0.0}


[websight]:   0%|          | 0/1001 [00:00<?, ?s/s]

10:58:07 | VLM_JUDGE       | JSON parse failed, trying regex. Raw: ```json
{
  "scores": {
    "A": 8,
    "B": 9,
    "C": 0,
    "D": 7
  },
  "ranking": [
    ["B"],
    ["A"],
    ["D"],
    ["C"]
  ]
}
</end_section_id|end_header_id

Certainly! I'll provide a co...
10:58:07 | VLM_JUDGE       | Extracted scores via regex: {'A': 8.0, 'B': 9.0, 'C': 0.0, 'D': 7.0}
11:05:17 | EVAL_PIPELINE   | VLM Judge error for websight_701_7d54559e: unhashable type: 'list'
11:06:43 | VLM_JUDGE       | JSON parse failed, trying regex. Raw: {
  "scores": {
    "A": 7,
    "B": 0,
    "C": 8,
    "D": 9
 9
  },
  "ranking": [
    ["D"],
    ["C"],
    ["A"],
    ["B"]
  ]
}...
11:06:43 | VLM_JUDGE       | Extracted scores via regex: {'A': 7.0, 'B': 0.0, 'C': 8.0, 'D': 9.0}
11:08:35 | VLM_JUDGE       | JSON parse failed, trying regex. Raw: ```json
{
  "scores": {
    "A": 9,
    "B": 6,
    "C": 5,
    "D": 2
  },
  "ranking": [
    ["A"],
    ["B"],
    ["C", "D"]
  ]
}
</end_header_id |assistant<

``

In [ ]:
# Cell 6: Verify Results
import pandas as pd

query = """
SELECT 
    r.model_name,
    COUNT(*) as total,
    ROUND(AVG(r.score_exact_match_normalized)::numeric, 3) as avg_em,
    ROUND(AVG(r.score_f1)::numeric, 3) as avg_f1,
    ROUND(AVG(e.glider_score)::numeric, 2) as avg_glider,
    ROUND(AVG(e.judge_molmo_score)::numeric, 2) as avg_vlm_judge,
    ROUND(AVG(e.judge_molmo_rank_group)::numeric, 2) as avg_rank
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE r.ok = true
GROUP BY r.model_name
ORDER BY avg_vlm_judge DESC NULLS LAST
"""

results_df = pd.read_sql(query, engine)
print("Results by Model:")
results_df

In [ ]:
# Cell 7: Check Coverage
coverage_query = """
SELECT 
    COUNT(*) as total_responses,
    SUM(CASE WHEN r.score_exact_match IS NOT NULL THEN 1 ELSE 0 END) as has_static,
    SUM(CASE WHEN r.confidence_score IS NOT NULL THEN 1 ELSE 0 END) as has_confidence,
    SUM(CASE WHEN e.glider_score IS NOT NULL THEN 1 ELSE 0 END) as has_glider,
    SUM(CASE WHEN e.judge_molmo_score IS NOT NULL THEN 1 ELSE 0 END) as has_vlm_judge
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE r.ok = true
"""

coverage_df = pd.read_sql(coverage_query, engine)
print("Metric Coverage:")
coverage_df.T